# City of Boston Subconscious Getting Started

This notebook will walk you through your first API calls with **Subconscious**, an integrated model and inference runtime that handles long context workflows through a simple API.

We'll build up to connecting an agent to the **City of Boston open data portal MCP** — no extra signup or configuration required.

Let's get you started in 2 simple steps:

1. Paste your API key below (Get one at [subconscious.dev/platform](https://www.subconscious.dev/platform))
2. Click "Run All" to jump right in, or run each cell one by one to see how it works.

Let's go!

In [ ]:
# You can find your API key on the platform:
# https://www.subconscious.dev/platform

# **Paste your API key here**
api_key = "sk-***"

In [ ]:
# Install the Subconscious Python SDK
# This is the only setup you'll need!

!pip install subconscious-sdk > /dev/null 2>&1

In [ ]:
# Connect to the Subconscious SDK in 2 lines of code.

from subconscious import Subconscious
client = Subconscious(api_key=api_key)

## *You're in.* Let's try it out!

The simplest thing you can do is give the agent an instruction and let it reason on its own.

Key parameters:
- **`engine`** — which model to use (we'll use **`"tim-claude"`**, Subconscious's Claude-backed compound engine. You can also experiment with our experimental **`tim`** engine or our **`tim-claude-heavy`** engine.)
- **`input.instructions`** — what you want the agent to do
- **`options.await_completion`** — wait for the full answer before returning

If you want to see what's going on behind the scenes, visit the [Subconscious Platform](https://www.subconscious.dev/runs) and click "View Runs" on the run you'd like to inspect.

In [ ]:
run = client.run(
    engine="tim-claude",
    input={
        "instructions": "What are some fun facts about the city of Boston?",
    },
    options={"await_completion": True},
)

print(run.result.answer)

## Using Tools

Agents become much more powerful when you give them **tools**. Subconscious ships with built-in search tools with zero setup.

Here are some popular ones to get started:

| Tool | API Name | What it does |
|------|----------|--------------|
| Fast Search | `fast_search` | Quick factual lookups |
| Web Search | `web_search` | Detailed web research |
| News Search | `news_search` | Search news articles |
| Page Reader | `page_reader` | Read a webpage URL |

To give an agent a tool, pass it along in the `tools` list.

In [ ]:
run = client.run(
    engine="tim-claude",
    input={
        "instructions": "What's happening in Boston this week? Give me a quick summary of local news.",
        "tools": [
            {"type": "platform", "id": "web_search"},
            {"type": "platform", "id": "news_search"},
        ],
    },
    options={"await_completion": True},
)

print(run.result.answer)

## Structured Output with Pydantic

Sometimes you don't want a free-text answer — you want **structured data** you can use in code (JSON with specific fields).

Subconscious supports this via [Pydantic](https://docs.pydantic.dev/) models. You define a schema and the agent's final answer will match it.

In [ ]:
from pydantic import BaseModel


class BostonNeighborhood(BaseModel):
    name: str
    known_for: str
    population_estimate: str


class BostonNeighborhoods(BaseModel):
    neighborhoods: list[BostonNeighborhood]


run = client.run(
    engine="tim-claude",
    input={
        "instructions": "List 5 Boston neighborhoods with what each is known for and a rough population estimate.",
        "answerFormat": BostonNeighborhoods,
    },
    options={"await_completion": True},
)

for n in run.result.parsed_answer["neighborhoods"]:
    print(f"{n['name']:20s} | {n['known_for']:40s} | Pop: {n['population_estimate']}")

## Connecting to Boston Open Data via MCP

Now let's connect the agent to a **real data source**. The City of Boston publishes an [MCP server](https://data-mcp.boston.gov/mcp) that gives the agent direct access to Boston's open data portal, no API key or extra setup required.

Just add `{"type": "mcp", "url": "..."}` to the tools list and the agent can query live MCP servers. We'll go ahead and add in the city of Boston MCP for this example.

In [ ]:
run = client.run(
    engine="tim-claude",
    input={
        "instructions": "How many restaurants are in Boston?",
        "tools": [
            {"type": "mcp", "url": "https://data-mcp.boston.gov/mcp"},
        ],
    },
    options={"await_completion": True},
)

print(run.result.answer)

## More complex interactions

Now that you're connected to the MCP, we can make a more complex instruction set. The best instructions tell the model what it's goals are, what pitfalls to look out for, and what output format you'd like to see.

The Boston City MCP has data sets for:
- Crime
- Population data
- Emergency Services
- The Bicycle network
- Utility Data
- Women-owned businesses
- and more!


In [ ]:
instructions='''
Goal: Given a hypothetical $200M capital budget, recommend which
BPS buildings to renovate vs replace vs close, optimizing for
students served per dollar and remaining useful life.

Pitfalls:
- Renovation cost is not in the dataset. Estimate from building
  size and condition score, and state your assumptions.
- Don't ignore enrollment trends. Replacing a building losing
  students is different from one at capacity.
- "Educational suitability" can't always be fixed with money.
  Call out where it's a layout problem vs a maintenance problem.

Output Format:
- 3 ranked lists (renovate / replace / close) with total cost
- Short justification per school
- Stated assumptions at the top
'''

run = client.run(
    engine="tim-claude",
    input={
        "instructions": instructions,
        "tools": [
            {"type": "mcp", "url": "https://data-mcp.boston.gov/mcp"},
        ],
    },
    options={"await_completion": True},
)

print(run.result.answer)

## Trying Different Engines

Subconscious offers several engines — each optimized for different trade-offs. Try swapping `engine` in any of the cells above!

| Engine | API Name | Best for |
|--------|----------|----------|
| **TIM-Claude** | `tim-claude` | **Compound reasoning with Claude** (used in this notebook) |
| TIM | `tim` | General-purpose on Subconscious's unified TIM stack |
| TIM-Edge | `tim-edge` | Speed and efficiency, search-heavy workloads |
| TIM-Claude-Heavy | `tim-claude-heavy` | Maximum reasoning with Claude Opus |
| TIM-OSS-Local | `tim-oss-local` | Tool-calling with TIM-trained OSS models |
| TIM-1.5 | `tim-1.5` | Tool-calling with large OSS models |

## What's Next?

You've now covered the core building blocks of Subconscious! Here are some ideas for what to explore next:

- **Custom tools** — connect the agent to your own HTTP endpoints ([docs](https://docs.subconscious.dev/core-concepts/tools))
- **MCP** — add any HTTP MCP server with `{"type": "mcp", "url": "..."}`; details and auth options [here](https://docs.subconscious.dev/core-concepts/tools#mcp-tools)
- **Streaming** — get results in real-time as the agent works ([streaming](https://docs.subconscious.dev/core-concepts/streaming#streaming))
- **Templates** — plug-and-play projects on the [platform](https://www.subconscious.dev/platform/templates)

### Useful Links

| Resource | Link |
|----------|------|
| Documentation | [docs.subconscious.dev](https://docs.subconscious.dev) |
| API Reference | [docs.subconscious.dev/api-reference](https://docs.subconscious.dev/api-reference/introduction) |
| Platform | [subconscious.dev/platform](https://www.subconscious.dev/platform) |

Happy building!